In [1]:
from pathlib import Path
import sys
sys.path.append(str(Path().resolve().parent / 'src'))

In [2]:
from typing import Any, List

from data_handlers.echr_data_handler import EchrDataHandler
from utils.evaluation_utils import EvaluationUtils
from utils.project_utils import ProjectUtils

import pandas as pd

In [3]:
project_root: Path = ProjectUtils.get_project_root(); project_root

PosixPath('/home/ssaha/Projects/redacted-text-utility')

In [4]:
echr_data_handler: EchrDataHandler = EchrDataHandler(project_root)
echr_raw_file_names: List[str] = echr_data_handler.get_available_raw_files()

In [5]:
k = 5
replacement_strategy="semantic_label_mask"

model_name = 'google--electra-large-discriminator'
data_dir_path = Path(f'/home/ssaha/model-checkpoints/echr/tc/{model_name}/additional-embeddings-none/sample-size-8435/data-fold-{k}')
model_file_path = data_dir_path / 'learning-rate-5e-7' / 'max-epochs-25' / 'mini-batch-size-2' / 'best-model.pt'

input_df = echr_data_handler.get_train_dev_test_datasetdict(k=k)["test"].to_pandas()
pe_df = echr_data_handler.get_private_entities_df(echr_raw_file_names[0])
id_column="itemid"
text_column="text"
class_column="binary_judgement"
pe_column="text_pe_ontonotes5_ner-english-ontonotes-large"
zero_entity_retain_text=True

result = EvaluationUtils.redact_and_evaluate_for_text_classifier(input_df=input_df,
                                                                 pe_df=pe_df,
                                                                 id_column=id_column,
                                                                 text_column=text_column,
                                                                 class_column=class_column,
                                                                 pe_column=pe_column,
                                                                 replacement_strategy=replacement_strategy,
                                                                 zero_entity_retain_text=zero_entity_retain_text,
                                                                 data_dir_path=data_dir_path,
                                                                 model_file_path=model_file_path)
print(result.detailed_results)

2026-01-15 16:19:12.073 | INFO     | data_handlers.echr_data_handler:get_dataframe_for_file:115 - Loading data from /home/ssaha/Projects/redacted-text-utility/data/raw/glnmario/ECHR/ECHR_Dataset.parquet


2026-01-15 16:19:45,790 Reading data from /home/ssaha/model-checkpoints/echr/tc/google--electra-large-discriminator/additional-embeddings-none/sample-size-8435/data-fold-5
2026-01-15 16:19:45,791 Train: /home/ssaha/model-checkpoints/echr/tc/google--electra-large-discriminator/additional-embeddings-none/sample-size-8435/data-fold-5/train.csv
2026-01-15 16:19:45,792 Dev: /home/ssaha/model-checkpoints/echr/tc/google--electra-large-discriminator/additional-embeddings-none/sample-size-8435/data-fold-5/dev.csv
2026-01-15 16:19:45,792 Test: /home/ssaha/model-checkpoints/echr/tc/google--electra-large-discriminator/additional-embeddings-none/sample-size-8435/data-fold-5/test_redacted_with_semantic_label_mask.csv
Check first row of test corpus:
Sentence[2092]: "The applicant is a Dutch construction company with limited liability having its registered seat in [GPE]. In the proceedings it is represented by Mr [PERSON], a lawyer practising in [GPE]. The facts of the case, as submitted by the partie

100%|██████████| 1687/1687 [06:45<00:00,  4.16it/s]


Results:
- F-score (micro) 0.8139
- F-score (macro) 0.8122
- Accuracy 0.8139

By class:
              precision    recall  f1-score   support

           1     0.7658    0.9053    0.8297       845
           0     0.8837    0.7221    0.7948       842

    accuracy                         0.8139      1687
   macro avg     0.8247    0.8137    0.8122      1687
weighted avg     0.8246    0.8139    0.8123      1687

